### Install dependencies if running remotely with tools like Google Colab


In [1]:
! pip install os
! pip install json
! pip install re
! pip install python-docx

ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os
ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json
ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re


### Imports

In [2]:
import os
import json
import re
import subprocess
from docx import Document

## Extrae los pactos de un documento .docx

⚠️ **AVISO**: ANTES SE PASARON MANUALMENTE los archivos `.doc` a `.docx`.

Esta función extrae los pactos de un documento `.docx` y los almacena en una lista de párrafos completos.

### Parámetros
- `doc_path` (`str`): Ruta al archivo `.docx`.

### Retorno
- `list[str]`: Lista de párrafos completos correspondientes a cada pacto.


In [4]:
def extraer_pactos(doc_path):

    
    doc = Document(doc_path)
    pactos = []
    pacto_actual = ""
    
    # Expresión regular para detectar títulos de pactos
    regex_pacto = re.compile(r"^(PRIMERO|PRIMERA|SEGUNDO|SEGUNDA|TERCERO|TERCERA|CUARTO|CUARTA|"
                             r"QUINTO|QUINTA|SEXTO|SEXTA|SÉPTIMO|SÉPTIMA|SEPTIMO|SEPTIMA|OCTAVO|OCTAVA|NOVENO|NOVENA|"
                             r"DECIMO|DECIMA|DÉCIMO|DÉCIMA|UNDÉCIMO|UNDÉCIMA|DECIMOPRIMERO|DECIMOPRIMERA|DUODÉCIMO|DUODÉCIMA|DUODECIMA|DUODECIMO|"
                             r"DECIMOSEGUNDO|DECIMOSEGUNDA|DECIMOTERCERO|DECIMOTERCERA|"
                             r"DECIMOCUARTO|DECIMOCUARTA|DECIMOQUINTO|DECIMOQUINTA|DECIMOSEXTO|DECIMOSEXTA|"
                             r"DECIMOSÉPTIMO|DECIMOSÉPTIMA|DECIMOCTAVO|DECIMOCTAVA|DECIMONOVENO|DECIMONOVENA|"
                             r"VIGÉSIMO|VIGÉSIMA|VIGESIMOPRIMERO|VIGESIMOPRIMERA|VIGESIMOSEGUNDO|VIGESIMOSEGUNDA|"
                             r"VIGESIMOTERCERO|VIGESIMOTERCERA|VIGESIMOCUARTO|VIGESIMOCUARTA|VIGESIMOQUINTO|VIGESIMOQUINTA|"
                             r"VIGESIMOSEXTO|VIGESIMOSEXTA|VIGESIMOSÉPTIMO|VIGESIMOSÉPTIMA|VIGESIMOOCTAVO|VIGESIMOOCTAVA|"
                             r"VIGESIMONOVENO|VIGESIMONOVENA|TRIGÉSIMO|TRIGÉSIMA)\.?", re.IGNORECASE)
    
    for paragraph in doc.paragraphs:
        texto = paragraph.text.strip()
        
        if not texto:
            continue  # Ignorar líneas vacías
        
        # Si el párrafo es un título de pacto, guardar el anterior y empezar uno nuevo
        if regex_pacto.match(texto):
            if pacto_actual:
                pactos.append({
                    "text": pacto_actual.strip(),
                    "label_gpt": {
                        "legalidad": "",
                        "abusividad": "",
                        "áreas de riesgo": "",
                        "vaguedades": "",
                        "lagunas": ""
                    },
                    "label_expected": {
                        "legalidad": "",
                        "abusividad": "",
                        "áreas de riesgo": "",
                        "vaguedades": "",
                        "lagunas": ""
                    }
                })
            pacto_actual = texto  # Empezar nuevo pacto
        else:
            pacto_actual += " " + texto
    
    # Guardar el último pacto si existe
    if pacto_actual:
        pactos.append({
            "text": pacto_actual.strip(),
            "label_gpt": {
                "legalidad": "",
                "abusividad": "",
                "áreas de riesgo": "",
                "vaguedades": "",
                "lagunas": ""
            },
            "label_expected": {
                "legalidad": "",
                "abusividad": "",
                "áreas de riesgo": "",
                "vaguedades": "",
                "lagunas": "" ""
            }
        })
    
    return pactos


## Procesa documentos .docx y extrae pactos en JSON

Esta función procesa todos los documentos `.docx` en un directorio y extrae los pactos en formato JSON.

### Parámetros
- `directorio` (`str`): Ruta de la carpeta con los documentos `.docx`.


In [5]:
def procesar_documentos(directorio):

    resultados = {}

    for archivo in os.listdir(directorio):
        if archivo.endswith(".docx"):
            ruta_completa = os.path.join(directorio, archivo)
            pactos = extraer_pactos(ruta_completa)
            resultados[archivo] = pactos

    # Guardar resultados en un archivo JSON
    with open("contratos.json", "w", encoding="utf-8") as json_file:
        json.dump(resultados, json_file, indent=4, ensure_ascii=False)

    print("✅ Los resultados están en 'contratos.json'.")



In [6]:
# Ruta donde están los archivos .docx
directorio_docx = "../dataset/contractExamples_org"

# Ejecutar la extracción
procesar_documentos(directorio_docx)

✅ Los resultados están en 'contratos.json'.


Funcion para fusionar json


In [24]:
import json

def añadir_label_a_redacciones(input_json, output_json):
    with open(input_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    for contrato, modelos in data.items():
        for modelo, contenido in modelos.items():
            # Si aún es solo un string (sin estructura), lo convertimos
            if isinstance(content := modelos[modelo], str):
                modelos[modelo] = {
                    "text": content,
                    "label": ""
                }

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f'Se añadió columna "label" por modelo en: {output_json}')


In [25]:
añadir_label_a_redacciones("redacciones_consolidadas.json", "redacciones_con_label.json")


Se añadió columna "label" por modelo en: redacciones_con_label.json


In [13]:
import json

def limpiar_etiquetas_incompletas(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels_requeridos = [
        "label_expected",
        "label_gpt4",
        "label_GPT4o",
        "label_GPT35",
        "label_DS"
    ]

    total_filtradas = 0

    for doc, clausulas in data.items():
        for i, clausula in enumerate(clausulas):
            tiene_todos = all(
                label in clausula and clausula[label] is not None
                for label in labels_requeridos
            )

            if not tiene_todos:
                # Dejamos solo el campo "text"
                texto = clausula.get("text", "")
                data[doc][i] = {"text": texto}
                total_filtradas += 1

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"✅ Clausulas limpiadas: {total_filtradas}")
    print(f"📝 Guardado en: {output_path}")


In [14]:
limpiar_etiquetas_incompletas("summary.json", "contratos_filtrados.json")


✅ Clausulas limpiadas: 62
📝 Guardado en: contratos_filtrados.json
